### Load ECSF as DF

In [4]:
# read ECSF Framework (ENISA) 
import pandas as pd 

excel_file = pd.ExcelFile('/mimer/NOBACKUP/groups/naiss2024-22-903/LLMedu/data/ECSF.xlsx')
print('sheet names: ',excel_file.sheet_names)
#print(len(excel_file.sheet_names[3:]))

sheet names:  ['Cover Page', 'Table Of Contents', 'Tasks', 'Knowledge', 'Skills', 'e-Competences', 'Deliverables']


In [14]:
# Read Knowledge first
df = pd.read_excel('/mimer/NOBACKUP/groups/naiss2024-22-903/LLMedu/data/ECSF.xlsx', sheet_name='Knowledge')
job_roles = df.iloc[0, 2:15].tolist() 
print('ECSF job roles: ',job_roles)
knowledge_by_role = {role: [] for role in job_roles} 

for row_idx in range(1, len(df)): 
    knowledge_statement = df.iloc[row_idx, 0] # 1

    # check what job roles are marked 
    for col_idx, role in enumerate(job_roles, start=2): 
        cell_value = df.iloc[row_idx, col_idx]

        # check if it is marked with an x 
        if isinstance(cell_value, str) and cell_value.strip().lower() == 'x': 
            knowledge_by_role[role].append(knowledge_statement)

# print the results for each job role 
for role, statements in knowledge_by_role.items(): 
    print(role, ' statements: ', statements)

ECSF job roles:  ['Chief Information Security Officer (CISO)', 'Cyber Incident Responder', 'Cyber Legal, Policy & Compliance Officer', 'Cyber Threat Intelligence Specialist', 'Cybersecurity Architect', 'Cybersecurity Auditor', 'Cybersecurity Educator', 'Cybersecurity Implementer', 'Cybersecurity Researcher', 'Cybersecurity Risk Manager', 'Digital Forensics Investigator', 'Penetration Tester']
Chief Information Security Officer (CISO)  statements:  [18, 19, 20, 21, 22, 24, 26, 33, 43, 56, 59]
Cyber Incident Responder  statements:  [4, 6, 7, 13, 14, 22, 26, 34, 35, 36, 37, 48, 63]
Cyber Legal, Policy & Compliance Officer  statements:  [19, 22, 24, 40, 53]
Cyber Threat Intelligence Specialist  statements:  [1, 4, 5, 10, 11, 12, 13, 14, 16, 26, 48, 57, 67]
Cybersecurity Architect  statements:  [13, 16, 21, 23, 24, 25, 26, 27, 29, 39, 40, 54, 55, 62, 64]
Cybersecurity Auditor  statements:  [2, 3, 8, 16, 24, 26, 40, 44]
Cybersecurity Educator  statements:  [15, 16, 17, 21, 22, 24, 26, 49]
Cy

In [15]:
unique_knowledge_statements = df.iloc[1:len(df), 1]
print(len(unique_knowledge_statements))
print(unique_knowledge_statements)

67
1           Advanced and persistent cyber threats (APT)
2      Auditing standards, methodologies and frameworks
3                        Auditing-related certification
4                            Computer networks security
5                                  Computer programming
                            ...                        
63            Secure Operation Centres (SOCs) operation
64               Security architecture reference models
65                                   Testing procedures
66      Testing standards, methodologies and frameworks
67    Threat actors Tactics, Techniques and Procedur...
Name: Unnamed: 1, Length: 67, dtype: object


### Label all KDs extracted from ECSF

In [17]:
# label the unique knowledge statements 
from datasets import Dataset 
KDs = pd.DataFrame({"Statement Description": unique_knowledge_statements})
KD2025 = Dataset.from_pandas(KDs, split='train')
print(KD2025)

# Load trained DistilBERT model 
from sklearn.metrics import multilabel_confusion_matrix
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import DataCollatorWithPadding
import sys
import torch
import numpy as np
sys.path.append("/mimer/NOBACKUP/groups/naiss2024-22-903/LLMedu")

# load model from a working checkpoint 
model_name='/mimer/NOBACKUP/groups/naiss2024-22-903/LLMedu/BERTv1final/checkpoint-430' #'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)             # the tokenizer is the same for all folds 
data_collator = DataCollatorWithPadding(tokenizer=tokenizer) # the data collator is the same for all folds 
# misc: (this includes Computer Science, Business and Law, Communication and Networking, Information Technology, Cyberspace Practice, Pedagogy, and Intelligence)
KAs = {"0": "miscellaneous",
           "1": "data security", 
           "2": "software security",
           "3": "component security", 
           "4": "connection security", 
           "5": "system security", 
           "6": "human security", 
           "7": "organizational security",
           "8": "societal security"}

def sigmoid(x):
   return 1/(1 + np.exp(-x))
    
model = AutoModelForSequenceClassification.from_pretrained(model_name, output_hidden_states=True,
                                                                        problem_type="multi_label_classification",
                                                                        num_labels=9, dropout=0.2)
model.eval()

Dataset({
    features: ['Statement Description'],
    num_rows: 67
})


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.2, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.2, inplace=False)


In [18]:
from transformers import AutoTokenizer
import datasets 
import re 
def clean_text(text): 
    special_chars = r'[\_/-]'
    cleaned_text = re.sub(special_chars,' ', text)                # replace - or _ or / with space 
    cleaned_text = re.sub(r'[^a-zA-Z0-9 -/()]', '', cleaned_text) # remove any non-alphanumeric /space character
    return cleaned_text.lower()  
    
def preprocess8(example):
    # preprocess and tokenize data 
    text = f"{example['Statement Description']}"       # our text to be labeled is the description of the KD
    text = text[13:]                                   # remove "Knowledge of" since it's the same for every KD
    text = clean_text(text)                            # remove special characters
    #one_hot = list(example.values())[1:9]              # read one-hot encoding 
    example = tokenizer(text, return_tensors='pt')                          # tokenize text 
    #example['labels'] = torch.tensor(one_hot).float()
    return example 
KD2025 = KD2025.map(preprocess8)
print(KD2025)

Parameter 'function'=<function preprocess8 at 0x7ff35248d040> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/67 [00:00<?, ? examples/s]

Dataset({
    features: ['Statement Description', 'input_ids', 'attention_mask'],
    num_rows: 67
})


In [20]:
llm_labels = [] 
tokenized_course = {}
KD2025_labels = {}
for i in range(len(KD2025)): 
    #print(course[str(i)])
    #tokenized_course[str(i)] = tokenizer(course[str(i)], return_tensors='pt') #, padding='max_length', max_length=64)
    row = KD2025[i] 
    print('KD: ',row['Statement Description'])

    # use CurricuLLM to label any new KDs 
    with torch.no_grad(): 
        predictions= model(input_ids=torch.tensor(row['input_ids']), 
                           attention_mask=torch.tensor(row['attention_mask']))#**KD2025[str(i)])
        prediction = sigmoid(predictions.logits).detach().cpu().numpy()
        predictions = (prediction > 0.5).astype(int).reshape(-1)
        if sum(predictions) == 0: 
                print('empty')
                out = np.argmax(prediction)
                prediction_zeros= np.zeros(predictions.shape)
                prediction_zeros[out] = 1 
                predictions = prediction_zeros.astype(int)
        llm_labels.append(predictions)
        
        print([i for i,x in enumerate(predictions) if x >0])
        KD2025_labels[i] = [i for i,x in enumerate(predictions) if x >0]

KD:  Advanced and persistent cyber threats (APT)
[7]
KD:  Auditing standards, methodologies and frameworks
[0]
KD:  Auditing-related certification
empty
[5]
KD:  Computer networks security
[5]
KD:  Computer programming
empty
[0]
KD:  Computer Security Incident Response Teams (CSIRTs) operation
empty
[7]
KD:  Computer systems vulnerabilities
[4]
KD:  Conformity assessment standards, methodologies and frameworks
empty
[0]
KD:  Criminal investigation procedures, standards, methodologies and frameworks
[7]
KD:  Cross-domain and border-domain knowledge related to cybersecurity
[7, 8]
KD:  Cyber threat actors
empty
[0]
KD:  Cyber Threat Intelligence (CTI) sharing standards, methodologies and frameworks
[0]
KD:  Cyber threats
empty
[1]
KD:  Cybersecurity attack procedures
empty
[5]
KD:  Cybersecurity awareness, education and training programme development
[0]
KD:  Cybersecurity controls and solutions
[5]
KD:  Cybersecurity education and training standards, methodologies and frameworks
[0]
KD:

In [38]:
# plot the distribution of KAs 
array = [i for i in KD2025_labels.values()]
array = [element for sublist in array for element in sublist]
print(array)

# Create histogram for 0-8 range with 11 bins (0, 1, 2, ..., 8)
bins = np.arange(-0.5, 9.5, 1)  # Bin edges from -0.5 to 8.5
hist, bin_edges = np.histogram(array, bins=bins)

print("\n=== HISTOGRAM ANALYSIS (0-8 range) ===")
print(f"Bin centers (statement counts): {list(range(9))}")
print(f"Frequency (number of roles): {list(hist)}")

# Calculate percentages
total_roles = len(array)
percentages = (hist / total_roles) * 100
cumulative = np.cumsum(hist)
cumulative_percent = np.cumsum(percentages)

print("\nDetailed frequency table:")
print("-" * 50)
print(f"{'Statements':>10} {'Frequency':>12} {'Percent':>10}")
print("-" * 50)
for i in range(len(hist)):
    print(f"{i:>10} {hist[i]:>12} {percentages[i]:>9.1f}% ")

print("-" * 50)
print(f"{'Total':>10} {total_roles:>12} {100:>9.1f}%")

[7, 0, 5, 5, 0, 7, 4, 0, 7, 7, 8, 0, 0, 1, 5, 0, 5, 0, 2, 5, 5, 0, 0, 7, 0, 0, 0, 2, 5, 0, 0, 5, 0, 4, 7, 0, 0, 0, 1, 0, 7, 7, 8, 7, 8, 5, 8, 7, 8, 7, 8, 7, 5, 5, 0, 5, 5, 0, 0, 0, 0, 0, 7, 0, 4, 7, 0, 4, 0, 0, 1, 4, 1]

=== HISTOGRAM ANALYSIS (0-8 range) ===
Bin centers (statement counts): [0, 1, 2, 3, 4, 5, 6, 7, 8]
Frequency (number of roles): [29, 4, 2, 0, 5, 13, 0, 14, 6]

Detailed frequency table:
--------------------------------------------------
Statements    Frequency    Percent
--------------------------------------------------
         0           29      39.7% 
         1            4       5.5% 
         2            2       2.7% 
         3            0       0.0% 
         4            5       6.8% 
         5           13      17.8% 
         6            0       0.0% 
         7           14      19.2% 
         8            6       8.2% 
--------------------------------------------------
     Total           73     100.0%


### Gather annotations for each job role

In [45]:
# now compute the distribution of KAs for every ECSF job role 
ENISA_roles = np.zeros((12,9))
count = 0 
for role, list_KDs in knowledge_by_role.items(): 
    print(role)
    #print(list_KDs)
    # subtract 1 
    wr_KD = [KD2025_labels[i-1] for i in list_KDs]
    # analyze the result 
    all_numbers = np.array([num for sublist in wr_KD for num in sublist]) 
    counts = np.bincount(all_numbers, minlength=9)
    prob_vector = counts / len(all_numbers)
    print(prob_vector)
    ENISA_roles[count,:] = prob_vector
    count += 1 

print(ENISA_roles)

Chief Information Security Officer (CISO)
[0.45454545 0.         0.09090909 0.         0.09090909 0.18181818
 0.         0.09090909 0.09090909]
Cyber Incident Responder
[0.46153846 0.15384615 0.         0.         0.07692308 0.23076923
 0.         0.07692308 0.        ]
Cyber Legal, Policy & Compliance Officer
[0.5        0.         0.         0.         0.         0.16666667
 0.         0.16666667 0.16666667]
Cyber Threat Intelligence Specialist
[0.28571429 0.14285714 0.         0.         0.         0.28571429
 0.         0.21428571 0.07142857]
Cybersecurity Architect
[0.47058824 0.05882353 0.05882353 0.         0.05882353 0.11764706
 0.         0.17647059 0.05882353]
Cybersecurity Auditor
[0.4 0.  0.  0.  0.  0.2 0.  0.2 0.2]
Cybersecurity Educator
[0.875 0.    0.    0.    0.    0.125 0.    0.    0.   ]
Cybersecurity Implementer
[0.36363636 0.09090909 0.         0.         0.18181818 0.27272727
 0.         0.09090909 0.        ]
Cybersecurity Researcher
[0.28571429 0.         0.    

### Match ENISA with NICE roles using the extracted KA distributions

In [44]:
NICE_roles = np.array([
    [0.09589041, 0.12328767, 0.05479452, 0.05479452, 0.05479452, 0.04109589, 0.08219178, 0.38356164, 0.10958904],
    [0.14583333, 0.06250000, 0.02083333, 0.00000000, 0.08333333, 0.02083333, 0.06250000, 0.43750000, 0.16666667],
    [0.18840580, 0.04347826, 0.02898551, 0.00000000, 0.07246377, 0.00000000, 0.04347826, 0.50724638, 0.11594203],
    [0.33333333, 0.06060606, 0.00000000, 0.00000000, 0.10606061, 0.01515152, 0.07575758, 0.28787879, 0.12121212],
    [0.26436782, 0.08045977, 0.00000000, 0.00000000, 0.17241379, 0.03448276, 0.08045977, 0.26436782, 0.10344828],
    [0.16949153, 0.11864407, 0.01694915, 0.00000000, 0.05084746, 0.00000000, 0.11864407, 0.35593220, 0.16949153],
    [0.08474576, 0.05084746, 0.01694915, 0.00000000, 0.15254237, 0.03389831, 0.05084746, 0.45762712, 0.15254237],
    [0.09638554, 0.10843373, 0.01204819, 0.01204819, 0.10843373, 0.03614458, 0.13253012, 0.31325301, 0.18072289],
    [0.17808219, 0.04109589, 0.04109589, 0.02739726, 0.06849315, 0.02739726, 0.04109589, 0.47945205, 0.09589041],
    [0.15625000, 0.04687500, 0.04687500, 0.03125000, 0.07812500, 0.03125000, 0.04687500, 0.45312500, 0.10937500],
    [0.17910448, 0.04477612, 0.04477612, 0.02985075, 0.07462687, 0.02985075, 0.04477612, 0.44776119, 0.10447761],
    [0.23417722, 0.08860759, 0.05696203, 0.02531646, 0.10759494, 0.05063291, 0.05696203, 0.31012658, 0.06962025],
    [0.15053763, 0.12903226, 0.03225806, 0.00000000, 0.07526882, 0.05376344, 0.06451613, 0.38709677, 0.10752688],
    [0.13669065, 0.07194245, 0.05755396, 0.04316547, 0.12230216, 0.05755396, 0.05035971, 0.38848921, 0.07194245],
    [0.06976744, 0.06976744, 0.02325581, 0.00000000, 0.09302326, 0.00000000, 0.06976744, 0.51162791, 0.16279070],
    [0.15686275, 0.05882353, 0.01960784, 0.00000000, 0.05882353, 0.00000000, 0.05882353, 0.50980392, 0.13725490],
    [0.16666667, 0.06944444, 0.09027778, 0.04166667, 0.15972222, 0.12500000, 0.04861111, 0.25000000, 0.04861111],
    [0.13761468, 0.06422018, 0.07339450, 0.04587156, 0.13761468, 0.12844037, 0.06422018, 0.28440367, 0.06422018],
    [0.12612613, 0.02702703, 0.18918919, 0.03603604, 0.10810811, 0.10810811, 0.03603604, 0.27927928, 0.09009009],
    [0.16666667, 0.08333333, 0.11805556, 0.03472222, 0.12500000, 0.07638889, 0.06250000, 0.26388889, 0.06944444],
    [0.12037037, 0.02777778, 0.19444444, 0.04629630, 0.10185185, 0.09259259, 0.03703704, 0.28703704, 0.09259259],
    [0.13636364, 0.09090909, 0.06363636, 0.03636364, 0.08181818, 0.08181818, 0.08181818, 0.33636364, 0.09090909],
    [0.13750000, 0.06250000, 0.08750000, 0.05000000, 0.07500000, 0.08750000, 0.06250000, 0.33750000, 0.10000000],
    [0.17582418, 0.07692308, 0.07692308, 0.05494505, 0.16483516, 0.06593407, 0.03296703, 0.25274725, 0.09890110],
    [0.05172414, 0.06896552, 0.00000000, 0.00000000, 0.06896552, 0.05172414, 0.05172414, 0.62068966, 0.08620690],
    [0.20000000, 0.12941176, 0.07058824, 0.01176471, 0.07058824, 0.08235294, 0.08235294, 0.24705882, 0.10588235],
    [0.22368421, 0.17105263, 0.00000000, 0.00000000, 0.06578947, 0.01315789, 0.11842105, 0.28947368, 0.11842105],
    [0.30769231, 0.07692308, 0.01923077, 0.00000000, 0.07692308, 0.00000000, 0.07692308, 0.30769231, 0.13461538],
    [0.19811321, 0.10377358, 0.03773585, 0.03773585, 0.20754717, 0.04716981, 0.06603774, 0.22641509, 0.07547170],
    [0.12371134, 0.08247423, 0.05154639, 0.04123711, 0.20618557, 0.07216495, 0.06185567, 0.27835052, 0.08247423],
    [0.17924528, 0.09433962, 0.05660377, 0.03773585, 0.14150943, 0.08490566, 0.05660377, 0.28301887, 0.06603774],
    [0.17647059, 0.10294118, 0.01470588, 0.00000000, 0.08823529, 0.04411765, 0.10294118, 0.36764706, 0.10294118],
    [0.17006803, 0.08843537, 0.06122449, 0.02721088, 0.20408163, 0.08163265, 0.04761905, 0.25850340, 0.06122449],
    [0.14035088, 0.22807018, 0.05263158, 0.03508772, 0.11403509, 0.08771930, 0.02631579, 0.21929825, 0.09649123],
    [0.10000000, 0.10000000, 0.01428571, 0.00000000, 0.18571429, 0.02857143, 0.10000000, 0.37142857, 0.10000000],
    [0.10769231, 0.10769231, 0.00000000, 0.00000000, 0.21538462, 0.03076923, 0.09230769, 0.33846154, 0.10769231],
    [0.14772727, 0.10227273, 0.00000000, 0.00000000, 0.07954545, 0.03409091, 0.04545455, 0.51136364, 0.07954545],
    [0.31868132, 0.05494505, 0.02197802, 0.00000000, 0.17582418, 0.03296703, 0.05494505, 0.26373626, 0.07692308],
    [0.08641975, 0.09876543, 0.03703704, 0.00000000, 0.13580247, 0.06172840, 0.08641975, 0.35802469, 0.13580247],
    [0.11764706, 0.17647059, 0.01470588, 0.00000000, 0.08823529, 0.04411765, 0.05882353, 0.35294118, 0.14705882],
    [0.13157895, 0.24561404, 0.05263158, 0.03508772, 0.09649123, 0.09649123, 0.03508772, 0.21052632, 0.09649123],
    [0.19672131, 0.01639344, 0.06557377, 0.03278689, 0.16393443, 0.09836066, 0.00000000, 0.40983607, 0.01639344]
])


In [48]:
# match the KA distribution of the ECSF job roles with that of the NICE framework 
# using the KL divergence or the Earth Mover's Distance or simply the MSE 

# do this both ways 
import numpy as np
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt

# Assuming you have two arrays: array1 and array2
# array1 shape: (n1, 9), array2 shape: (n2, 9)

# Example arrays (replace with your actual data)
# array1 = np.random.rand(42, 9)  # Your first array
# array2 = np.random.rand(42, 9)  # Your second array

def find_most_similar_rows(array1, array2, metric='euclidean', top_k=5):
    """
    Find the most similar rows between two arrays.
    
    Parameters:
    - array1: First array (n1 x m)
    - array2: Second array (n2 x m)
    - metric: Distance metric ('euclidean', 'cosine', 'cityblock', etc.)
    - top_k: Number of top matches to return
    
    Returns:
    - distances: Distance matrix (n1 x n2)
    - top_matches: List of (idx1, idx2, distance) for top_k matches
    """
    
    # Compute pairwise distance matrix
    distances = cdist(array1, array2, metric=metric)
    
    # Find top_k smallest distances
    n1, n2 = distances.shape
    flat_indices = np.argsort(distances.ravel())[:top_k]
    
    # Convert flat indices to row/col indices
    idx1 = flat_indices // n2
    idx2 = flat_indices % n2
    
    top_matches = []
    for i, j in zip(idx1, idx2):
        top_matches.append((i, j, distances[i, j]))
    
    return distances, top_matches

# Usage
metric='cosine'
distances, top_matches = find_most_similar_rows(ENISA_roles, NICE_roles, metric=metric, top_k=10)

print("Top 10 most similar row pairs:")
print("-" * 60)
print(f"{'Array1 Row':<12} {'Array2 Row':<12} {'Distance':<15} {'Similarity %':<15}")
print("-" * 60)
for idx1, idx2, dist in top_matches:
    # Convert distance to similarity percentage (for cosine: similarity = 1 - distance)
    similarity = (1 - dist) * 100 if metric == 'cosine' else 100 / (1 + dist)
    print(f"{idx1:<12} {idx2:<12} {dist:<15.6f} {similarity:<15.2f}%")

Top 10 most similar row pairs:
------------------------------------------------------------
Array1 Row   Array2 Row   Distance        Similarity %   
------------------------------------------------------------
9            37           0.024835        97.52          %
9            4            0.024864        97.51          %
9            28           0.040915        95.91          %
9            11           0.043497        95.65          %
9            30           0.044110        95.59          %
9            32           0.044513        95.55          %
9            23           0.047415        95.26          %
9            3            0.051317        94.87          %
9            16           0.066666        93.33          %
9            27           0.068757        93.12          %


In [49]:
import numpy as np
from scipy.spatial.distance import cdist

# Assuming you have two arrays: array1 and array2
# array1 shape: (n1, 9), array2 shape: (n2, 9)
array1 = ENISA_roles
array2 = NICE_roles
def find_best_matches_bidirectional(array1, array2, metric='cosine'):
    """
    Find the best matching rows between two arrays in both directions.
    
    Parameters:
    - array1: First array (n1 x m)
    - array2: Second array (n2 x m)
    - metric: Distance metric ('euclidean', 'cosine', 'cityblock', etc.)
    
    Returns:
    - array1_to_array2: For each row in array1, best match in array2
    - array2_to_array1: For each row in array2, best match in array1
    - distances: Distance matrix (n1 x n2)
    """
    
    # Compute pairwise distance matrix
    distances = cdist(array1, array2, metric=metric)
    
    # For each row in array1, find best match in array2 (smallest distance)
    array1_to_array2 = []
    for i in range(len(array1)):
        best_match_idx = np.argmin(distances[i])
        best_distance = distances[i, best_match_idx]
        array1_to_array2.append({
            'array1_row': i,
            'best_array2_match': best_match_idx,
            'distance': best_distance,
            'similarity': 1 - best_distance if metric == 'cosine' else None
        })
    
    # For each row in array2, find best match in array1 (smallest distance)
    # Note: we need to look at columns of the distance matrix
    array2_to_array1 = []
    for j in range(len(array2)):
        best_match_idx = np.argmin(distances[:, j])  # Min in column j
        best_distance = distances[best_match_idx, j]
        array2_to_array1.append({
            'array2_row': j,
            'best_array1_match': best_match_idx,
            'distance': best_distance,
            'similarity': 1 - best_distance if metric == 'cosine' else None
        })
    
    return array1_to_array2, array2_to_array1, distances

# Usage with your data
array1_to_array2, array2_to_array1, distances = find_best_matches_bidirectional(
    array1, array2, metric='cosine'
)

# Print results
print("=" * 80)
print("BEST MATCHES: Array1 → Array2")
print("=" * 80)
print(f"{'Array1 Row':<12} {'Best Array2 Match':<20} {'Distance':<15} {'Similarity %':<15}")
print("-" * 80)
for match in array1_to_array2:
    similarity_pct = (1 - match['distance']) * 100  # Convert to percentage
    print(f"{match['array1_row']:<12} {match['best_array2_match']:<20} {match['distance']:<15.6f} {similarity_pct:<15.2f}")

print("\n" + "=" * 80)
print("BEST MATCHES: Array2 → Array1")
print("=" * 80)
print(f"{'Array2 Row':<12} {'Best Array1 Match':<20} {'Distance':<15} {'Similarity %':<15}")
print("-" * 80)
for match in array2_to_array1:
    similarity_pct = (1 - match['distance']) * 100  # Convert to percentage
    print(f"{match['array2_row']:<12} {match['best_array1_match']:<20} {match['distance']:<15.6f} {similarity_pct:<15.2f}")

BEST MATCHES: Array1 → Array2
Array1 Row   Best Array2 Match    Distance        Similarity %   
--------------------------------------------------------------------------------
0            37                   0.175949        82.41          
1            37                   0.227761        77.22          
2            3                    0.141901        85.81          
3            25                   0.156496        84.35          
4            37                   0.105836        89.42          
5            3                    0.139435        86.06          
6            3                    0.306819        69.32          
7            37                   0.209298        79.07          
8            1                    0.071392        92.86          
9            37                   0.024835        97.52          
10           16                   0.280159        71.98          
11           16                   0.430040        57.00          

BEST MATCHES: Array2 → Array1


In [55]:
import numpy as np
from scipy.special import kl_div, rel_entr
from scipy.stats import entropy
from scipy.spatial.distance import cdist

def kl_divergence_bidirectional(array1, array2, epsilon=1e-10):
    """
    Compute KL divergence between rows of two arrays.
    KL(P||Q) measures information lost when Q is used to approximate P.
    
    Note: KL divergence is asymmetric! KL(P||Q) ≠ KL(Q||P)
    """
    # Add small epsilon to avoid log(0)
    array1_safe = array1 + epsilon
    array2_safe = array2 + epsilon
    
    # Normalize to ensure they sum to 1 (just in case)
    array1_norm = array1_safe / array1_safe.sum(axis=1, keepdims=True)
    array2_norm = array2_safe / array2_safe.sum(axis=1, keepdims=True)
    
    n1, n2 = len(array1), len(array2)
    
    # Compute KL divergence matrix: KL(array1[i] || array2[j])
    kl_matrix_forward = np.zeros((n1, n2))
    for i in range(n1):
        for j in range(n2):
            # KL(P||Q) = Σ P(x) * log(P(x)/Q(x))
            kl_matrix_forward[i, j] = entropy(array1_norm[i], array2_norm[j])
    
    # Compute reverse KL: KL(array2[j] || array1[i])
    kl_matrix_reverse = np.zeros((n2, n1))
    for j in range(n2):
        for i in range(n1):
            kl_matrix_reverse[j, i] = entropy(array2_norm[j], array1_norm[i])
    
    # Find best matches in both directions
    # Forward: For each array1 row, find array2 row with smallest KL divergence
    array1_to_array2 = []
    for i in range(n1):
        best_match_idx = np.argmin(kl_matrix_forward[i])
        best_kl = kl_matrix_forward[i, best_match_idx]
        array1_to_array2.append({
            'array1_row': i,
            'best_array2_match': best_match_idx,
            'kl_divergence': best_kl,
            'is_symmetric': kl_matrix_reverse[best_match_idx, i] == best_kl
        })
    
    # Reverse: For each array2 row, find array1 row with smallest KL divergence
    array2_to_array1 = []
    for j in range(n2):
        best_match_idx = np.argmin(kl_matrix_reverse[j])
        best_kl = kl_matrix_reverse[j, best_match_idx]
        array2_to_array1.append({
            'array2_row': j,
            'best_array1_match': best_match_idx,
            'kl_divergence': best_kl,
            'is_symmetric': kl_matrix_forward[best_match_idx, j] == best_kl
        })
    
    return {
        'kl_matrix_forward': kl_matrix_forward,  # KL(array1||array2)
        'kl_matrix_reverse': kl_matrix_reverse,  # KL(array2||array1)
        'array1_to_array2': array1_to_array2,
        'array2_to_array1': array2_to_array1
    }

# Alternative: Symmetric Jensen-Shannon Divergence
def jensen_shannon_divergence_bidirectional(array1, array2, epsilon=1e-10):
    """
    Jensen-Shannon Divergence: Symmetric version of KL divergence.
    JS(P||Q) = 0.5 * [KL(P||M) + KL(Q||M)] where M = 0.5*(P+Q)
    
    Returns values between 0 and 1 (or log2 for 0 to log(2))
    """
    # Add epsilon and normalize
    array1_safe = array1 + epsilon
    array2_safe = array2 + epsilon
    array1_norm = array1_safe / array1_safe.sum(axis=1, keepdims=True)
    array2_norm = array2_safe / array2_safe.sum(axis=1, keepdims=True)
    
    n1, n2 = len(array1), len(array2)
    js_matrix = np.zeros((n1, n2))
    
    for i in range(n1):
        p = array1_norm[i]
        for j in range(n2):
            q = array2_norm[j]
            # Compute M = average distribution
            m = 0.5 * (p + q)
            # JS divergence
            js = 0.5 * (entropy(p, m) + entropy(q, m))
            js_matrix[i, j] = js
    
    # Find best matches
    array1_to_array2 = []
    for i in range(n1):
        best_match_idx = np.argmin(js_matrix[i])
        array1_to_array2.append({
            'array1_row': i,
            'best_array2_match': best_match_idx,
            'js_divergence': js_matrix[i, best_match_idx]
        })
    
    array2_to_array1 = []
    for j in range(n2):
        best_match_idx = np.argmin(js_matrix[:, j])
        array2_to_array1.append({
            'array2_row': j,
            'best_array1_match': best_match_idx,
            'js_divergence': js_matrix[best_match_idx, j]
        })
    
    return {
        'js_matrix': js_matrix,
        'array1_to_array2': array1_to_array2,
        'array2_to_array1': array2_to_array1
    }

out = kl_divergence_bidirectional(array1, array2, epsilon=1e-10)
print([d['best_array2_match'] for d in out['array1_to_array2']])
out2 = kl_divergence_bidirectional(array2, array1, epsilon=1e-10)
print([d['best_array2_match'] for d in out2['array1_to_array2']])

[16, 25, 37, 25, 37, 25, 37, 16, 2, 4, 16, 16]
[4, 4, 4, 9, 9, 4, 9, 9, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 9, 4, 9, 4, 4, 4, 4, 4, 4, 4, 9, 9, 9, 4, 4, 9, 4, 4]


In [61]:
from scipy.stats import wasserstein_distance
from scipy.optimize import linear_sum_assignment

def earth_movers_distance_bidirectional(array1, array2):
    """
    Compute Earth Mover's Distance (Wasserstein distance) between rows.
    
    EMD measures the minimum cost to transform one distribution into another.
    For 1D distributions (like your 9-dimensional vectors), we can compute directly.
    """
    n1, n2 = len(array1), len(array2)
    
    # Create positions for each dimension (0, 1, 2, ..., 8)
    positions = np.arange(array1.shape[1])
    
    # Compute EMD matrix
    emd_matrix = np.zeros((n1, n2))
    for i in range(n1):
        for j in range(n2):
            # Wasserstein distance for 1D distributions
            emd_matrix[i, j] = wasserstein_distance(positions, positions, 
                                                   array1[i], array2[j])
    
    # Find best matches
    array1_to_array2 = []
    for i in range(n1):
        best_match_idx = np.argmin(emd_matrix[i])
        array1_to_array2.append({
            'array1_row': i,
            'best_array2_match': best_match_idx,
            'emd': emd_matrix[i, best_match_idx]
        })
    
    array2_to_array1 = []
    for j in range(n2):
        best_match_idx = np.argmin(emd_matrix[:, j])
        array2_to_array1.append({
            'array2_row': j,
            'best_array1_match': best_match_idx,
            'emd': emd_matrix[best_match_idx, j]
        })
    
    return {
        'emd_matrix': emd_matrix,
        'array1_to_array2': array1_to_array2,
        'array2_to_array1': array2_to_array1
    }
print('ENISA to NICE')
print('emd')
out = earth_movers_distance_bidirectional(array1, array2)
print([d['best_array2_match'] for d in out['array1_to_array2']])
print('KL')
out2 = kl_divergence_bidirectional(array1, array2, epsilon=1e-10)
print([d['best_array2_match'] for d in out2['array1_to_array2']])
print('JS')
out3 = jensen_shannon_divergence_bidirectional(array1, array2, epsilon=1e-10)
print([d['best_array2_match'] for d in out3['array1_to_array2']])

print('NICE to ENISA')
print('emd')
out = earth_movers_distance_bidirectional(array2, array1)
print([d['best_array2_match'] for d in out['array1_to_array2']])
print('KL')
out2 = kl_divergence_bidirectional(array2, array1, epsilon=1e-10)
print([d['best_array2_match'] for d in out2['array1_to_array2']])
print('JS')
out3 = jensen_shannon_divergence_bidirectional(array2, array1, epsilon=1e-10)
print([d['best_array2_match'] for d in out3['array1_to_array2']])

ENISA to NICE
emd
[40, 37, 37, 37, 37, 3, 37, 37, 15, 4, 28, 37]
KL
[16, 25, 37, 25, 37, 25, 37, 16, 2, 4, 16, 16]
JS
[37, 37, 3, 25, 37, 3, 37, 37, 2, 4, 16, 16]
NICE to ENISA
emd
[8, 8, 8, 5, 9, 8, 8, 8, 8, 8, 8, 9, 9, 9, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 8, 9, 9, 5, 9, 9, 9, 8, 9, 9, 8, 9, 8, 9, 8, 8, 9, 9]
KL
[4, 4, 4, 9, 9, 4, 9, 9, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 9, 4, 9, 4, 4, 4, 4, 4, 4, 4, 9, 9, 9, 4, 4, 9, 4, 4]
JS
[9, 9, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 9, 9, 4, 9, 4, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9]


In [ ]:
# Also print the names of the titles 

In [58]:
# Also do the Tasks and SKills to see if that changes anything. 

### compare this to bag of words 

### Or a comparison with BERT features? 
Bert = Tokenizer, Embedding, Encoder + Task Head. 
Maybe there's a way to extract encodings and use those to match the roles. 

Should BERT be trained or untrained? 